# <b><font color='forestgreen'>Polars: сложные фильтры и группировка</font></b>

### <font color='forestgreen'>1. Обработка пропусков</font>

In [2]:
# мпорт библиотек
import polars as pl

In [45]:
# Запишем данные из файла books_data_short.csv в dataframe df_books
df_books = pl.read_csv('../data/books_data_short.csv', null_values = 'NaN')
df_books

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,null,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,null,"""magazine"""


In [44]:
df_books.describe()

statistic,price,type,year,conditional
str,f64,str,f64,f64
"""count""",10.0,"""10""",10.0,10.0
"""null_count""",0.0,"""0""",0.0,0.0
"""mean""",1461.034247,null,2003.6,0.1
"""std""",1600.839522,null,8.884443,null
"""min""",249.415919,"""book""",1991.0,0.0
"""25%""",605.61386,null,1996.0,null
"""50%""",1211.564183,null,2006.0,null
"""75%""",1370.04374,null,2008.0,null
"""max""",5849.487399,"""newspaper""",2021.0,1.0


##### <font color='forestgreen'>1.1. Определение недостающих данных</font>

In [4]:
# Проверка в столбце
df_books['year'].is_null().sum()

2

In [5]:
# Проверка в столбце
df_books['year'].null_count()

2

In [6]:
# Проверка в DataFrame
df_books.null_count()

price,year,type
u32,u32,u32
0,2,0


In [46]:
df_books.filter(pl.col('year').is_null())

price,year,type
f64,f64,str
5849.487399,null,"""magazine"""
605.61386,null,"""magazine"""


##### <font color='forestgreen'>1.2. Удаление пропущенных данных</font>

In [7]:
# Удаление всех пропущенных значений
df_books.drop_nulls() # Создает копию

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
1370.04374,2008.0,"""magazine"""


In [51]:
# Удаление колонок со всеми NaN
df_books[[s.name for s in df_books if not (s.null_count() == df_books.height)]]

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,null,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,null,"""magazine"""


In [9]:
for s in df_books:
    print(s.name)

price
year
type


##### <font color='forestgreen'>1.3. Заполнение пропущенных данных</font>
df.fill_null([константа, 'forward', 'backward', 'min', 'max', 'zero', 'one', 'mean'])

In [64]:
# Заполнение константой
df_books_filled = df_books.fill_null('unknown')
df_books_filled

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,null,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,null,"""magazine"""


In [11]:
# Заполнение предыдущим значением
df_books.fill_null(strategy = 'forward')

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,1991.0,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,1996.0,"""magazine"""


##### <font color='forestgreen'>1.4. Прямое и обратное заполнение</font>

In [12]:
df_books.with_columns(pl.col('year').backward_fill())

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,2021.0,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,2008.0,"""magazine"""


##### <font color='forestgreen'>1.5. нтерполяция</font>
нтерполяция подразумевает оценку пропущенных значений на основе соседних точек данных.

In [13]:
df_books = df_books.interpolate()
df_books

price,year,type
f64,f64,str
748.022957,1991.0,"""encyclopedia"""
5849.487399,2006.0,"""magazine"""
546.485157,2021.0,"""book"""
1130.831881,1993.0,"""magazine"""
1247.822382,2002.0,"""magazine"""
1211.564183,2008.0,"""magazine"""
1651.054987,2009.0,"""newspaper"""
249.415919,1996.0,"""encyclopedia"""
605.61386,2002.0,"""magazine"""


### <font color='forestgreen'>2. зменение типов данных (кастинг)</font>
[DateTypes](https://docs.pola.rs/api/python/stable/reference/datatypes.html)

In [14]:
df_books['year'].cast(pl.Int32)

year
i32
1991
2006
2021
1993
2002
2008
2009
1996
2002


In [15]:
df_books = df_books.with_columns(
    pl.col('year').cast(pl.Int32)
)

In [16]:
# Переименование столбцов 
df_books.rename({'price':'Price'})

Price,year,type
f64,i32,str
748.022957,1991,"""encyclopedia"""
5849.487399,2006,"""magazine"""
546.485157,2021,"""book"""
1130.831881,1993,"""magazine"""
1247.822382,2002,"""magazine"""
1211.564183,2008,"""magazine"""
1651.054987,2009,"""newspaper"""
249.415919,1996,"""encyclopedia"""
605.61386,2002,"""magazine"""


### <font color='forestgreen'>3. Операции агрегации</font>

In [17]:
df_books['price'].sum()

14610.342465841382

In [18]:
df_books.select([
    pl.col("price").count().alias("count"),
    pl.col("price").null_count().alias("null_count"),
    pl.col("price").mean().alias("mean"),
    pl.col("price").std().alias("std_dev"),
    pl.col("price").median().alias("median"),
    pl.col("price").min().alias("min"),
    pl.col("price").quantile(0.25).alias("25%"),
    pl.col("price").quantile(0.5).alias("50%"),
    pl.col("price").quantile(0.75).alias("75%"),
    pl.col("price").max().alias("max"),
])

count,null_count,mean,std_dev,median,min,25%,50%,75%,max
u32,u32,f64,f64,f64,f64,f64,f64,f64,f64
10,0,1461.034247,1600.839522,1171.198032,249.415919,605.61386,1211.564183,1370.04374,5849.487399


### <font color='forestgreen'>4. Группировка</font>

In [19]:
# group_by
df_books.group_by('year').len() #вместо count

year,len
i32,u32
2009,1
1993,1
2008,2
1996,1
2021,1
2002,2
1991,1
2006,1


In [20]:
# несколько функций аггрегаций
df_books.group_by('type').agg(
    [
        pl.mean('price').alias('mean_price'),
        pl.median('year').alias('median_year'),
        pl.len().alias('count')
    ])

type,mean_price,median_year,count
str,f64,f64,u32
"""magazine""",1902.560574,2004.0,6
"""book""",546.485157,2021.0,1
"""encyclopedia""",498.719438,1993.5,2
"""newspaper""",1651.054987,2009.0,1


In [21]:
df_books.pivot("type", index="year", values="price", aggregate_function = 'mean')

year,encyclopedia,magazine,book,newspaper
i32,f64,f64,f64,f64
1991,748.022957,null,null,null
2006,null,5849.487399,null,null
2021,null,null,546.485157,null
1993,null,1130.831881,null,null
2002,null,926.718121,null,null
2008,null,1290.803962,null,null
2009,null,null,null,1651.054987
1996,249.415919,null,null,null


### <font color='forestgreen'>5. Оконные функции</font>

<b>Базовый синтаксис: </b>
df.with_columns( pl.col("target_column").agg_function().over("group_column") )

- <i>pl.col("target_column")</i> – столбец, к которому применяется агрегационная функция.
- <i>agg_function()</i> – агрегационная функция (min, max, sum, mean и т.д.).
- <i>over("group_column")</i> – столбец (или список столбцов), по которому создается окно.

In [22]:
df_books.with_columns([
    pl.col("price").mean().over('type').alias("mean_price_by_type"),
    pl.col("price").max().over('type').alias("max_price_by_type")
])

price,year,type,mean_price_by_type,max_price_by_type
f64,i32,str,f64,f64
748.022957,1991,"""encyclopedia""",498.719438,748.022957
5849.487399,2006,"""magazine""",1902.560574,5849.487399
546.485157,2021,"""book""",546.485157,546.485157
1130.831881,1993,"""magazine""",1902.560574,5849.487399
1247.822382,2002,"""magazine""",1902.560574,5849.487399
1211.564183,2008,"""magazine""",1902.560574,5849.487399
1651.054987,2009,"""newspaper""",1651.054987,1651.054987
249.415919,1996,"""encyclopedia""",498.719438,748.022957
605.61386,2002,"""magazine""",1902.560574,5849.487399


In [23]:
# Вычислим ранк каждого издания по году внутри типа издания
df_books.with_columns([
    pl.col("year").rank('dense', descending = False).over("type").alias("rank_year_by_type")
]).sort(by = ['type', "rank_year_by_type"])

price,year,type,rank_year_by_type
f64,i32,str,u32
546.485157,2021,"""book""",1
748.022957,1991,"""encyclopedia""",1
249.415919,1996,"""encyclopedia""",2
1130.831881,1993,"""magazine""",1
1247.822382,2002,"""magazine""",2
605.61386,2002,"""magazine""",2
5849.487399,2006,"""magazine""",3
1211.564183,2008,"""magazine""",4
1370.04374,2008,"""magazine""",4


In [59]:
# Вычислите ранг каждого издания по стоимости (чем выше стоимость - тем выше ранг) внутри каждого типа
df_books.with_columns([
    pl.col("price").rank('dense', descending = True).over("type").alias("rank_price_by_type")
]).sort(by = ['type', "rank_price_by_type"])    


price,year,type,rank_price_by_type
f64,f64,str,u32
546.485157,2021.0,"""book""",1
748.022957,1991.0,"""encyclopedia""",1
249.415919,1996.0,"""encyclopedia""",2
5849.487399,null,"""magazine""",1
1370.04374,2008.0,"""magazine""",2
1247.822382,2002.0,"""magazine""",3
1211.564183,2008.0,"""magazine""",4
1130.831881,1993.0,"""magazine""",5
605.61386,null,"""magazine""",6


### <font color='forestgreen'>6. Уникальные значения</font>

In [25]:
# Способ №1
df_books['price'].n_unique()

10

In [60]:
# Способ №2
df_books.select(pl.approx_n_unique("type")) #HyperLogLog++

type
u32
4


### <font color='forestgreen'>7. Сложная фильтрация</font>

In [27]:
df_books.filter((pl.col('year') > 2002) & (pl.col('type') == 'magazine'))

price,year,type
f64,i32,str
5849.487399,2006,"""magazine"""
1211.564183,2008,"""magazine"""
1370.04374,2008,"""magazine"""


In [28]:
df_books.filter(~(pl.col('type') == 'encyclopedia')) # аналог NOT
df_books.filter(pl.col('type') != 'encyclopedia')

price,year,type
f64,i32,str
5849.487399,2006,"""magazine"""
546.485157,2021,"""book"""
1130.831881,1993,"""magazine"""
1247.822382,2002,"""magazine"""
1211.564183,2008,"""magazine"""
1651.054987,2009,"""newspaper"""
605.61386,2002,"""magazine"""
1370.04374,2008,"""magazine"""


##### <font color='forestgreen'>7.1. спользование функций для фильтрации</font>

In [29]:
def expensive(price):
    return price >= 5000

df_books.filter(expensive(pl.col('price'))) 

price,year,type
f64,i32,str
5849.487399,2006,"""magazine"""


##### <font color='forestgreen'>7.2. Цепочка фильтров</font>

In [30]:
df_books.filter(pl.col('price') > 1000).filter(pl.col('year') > 2000)

price,year,type
f64,i32,str
5849.487399,2006,"""magazine"""
1247.822382,2002,"""magazine"""
1211.564183,2008,"""magazine"""
1651.054987,2009,"""newspaper"""
1370.04374,2008,"""magazine"""


### <font color='forestgreen'>8. Условия</font>
Polars поддерживает условия типа if-else в выражениях с синтаксисом when, then, otherwise. Предикат помещается в выражение when, и когда он оценивается как true, применяется выражение then, в противном случае применяется выражение otherwise (по порядку).

In [31]:
df_books = df_books.select(
    pl.col('price'),
    pl.col('type'),
    pl.col('year'),
    pl.when(pl.col('year') > 2014)
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias('conditional')
)
df_books

price,type,year,conditional
f64,str,i32,bool
748.022957,"""encyclopedia""",1991,false
5849.487399,"""magazine""",2006,false
546.485157,"""book""",2021,true
1130.831881,"""magazine""",1993,false
1247.822382,"""magazine""",2002,false
1211.564183,"""magazine""",2008,false
1651.054987,"""newspaper""",2009,false
249.415919,"""encyclopedia""",1996,false
605.61386,"""magazine""",2002,false


### <font color='forestgreen'>9. Задача</font>

In [3]:
# Загрузка данных
# import seaborn as sns
df_penguins = pl.DataFrame(sns.load_dataset('penguins'))
df_penguins

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",null,null,null,null,null
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""
…,…,…,…,…,…,…
"""Gentoo""","""Biscoe""",null,null,null,null,null
"""Gentoo""","""Biscoe""",46.8,14.3,215.0,4850.0,"""Female"""
"""Gentoo""","""Biscoe""",50.4,15.7,222.0,5750.0,"""Male"""


In [4]:
# Посчитать количество пропусков по каждому столбцу.
df_penguins.describe()
df_penguins.null_count()

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
u32,u32,u32,u32,u32,u32,u32
0,0,2,2,2,2,11


In [5]:
# Заполнить пропуски в bill_length_mm и bill_depth_mm средним значением по виду (species).
# Пропуски в sex заполнить "Unknown".
df_penguins_null = df_penguins.with_columns([
    pl.col("bill_length_mm").fill_null(strategy='mean'),
    pl.col("bill_depth_mm").fill_null(strategy='mean'),
    pl.col("sex").fill_null("Unknown")
])
df_penguins_null

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female"""
"""Adelie""","""Torgersen""",43.92193,17.15117,null,null,"""Unknown"""
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female"""
…,…,…,…,…,…,…
"""Gentoo""","""Biscoe""",43.92193,17.15117,null,null,"""Unknown"""
"""Gentoo""","""Biscoe""",46.8,14.3,215.0,4850.0,"""Female"""
"""Gentoo""","""Biscoe""",50.4,15.7,222.0,5750.0,"""Male"""


In [6]:
# Посчитать среднюю массу тела (body_mass_g) и среднюю длину плавника (flipper_length_mm) по виду и полу.
df_penguins_mean = df_penguins.group_by(['species', 'sex']).agg([
    pl.mean("body_mass_g").alias("mean_body_mass_g"),
    pl.mean("flipper_length_mm").alias("mean_flipper_length_mm")
])
df_penguins_mean

species,sex,mean_body_mass_g,mean_flipper_length_mm
str,str,f64,f64
"""Chinstrap""","""Male""",3938.970588,199.911765
"""Adelie""","""Female""",3368.835616,187.794521
"""Gentoo""","""Male""",5484.836066,221.540984
"""Gentoo""",null,4587.5,215.75
"""Gentoo""","""Female""",4679.741379,212.706897
"""Chinstrap""","""Female""",3527.205882,191.735294
"""Adelie""","""Male""",4043.493151,192.410959
"""Adelie""",null,3540.0,185.6


In [7]:
# Сделать pivot-таблицу: индекс — species, колонки — sex, значения — средняя масса тела (body_mass_g).
df_penguins.pivot(index="species", columns="sex", values="body_mass_g", aggregate_function = 'mean')


C:\Users\Vitaliy\AppData\Local\Temp\ipykernel_26768\1279533458.py:2: DeprecationWarning: the argument `columns` for `DataFrame.pivot` is deprecated. It was renamed to `on` in version 1.0.0.
  df_penguins.pivot(index="species", columns="sex", values="body_mass_g", aggregate_function = 'mean')


species,Male,Female,null
str,f64,f64,f64
"""Adelie""",4043.493151,3368.835616,3540.0
"""Chinstrap""",3938.970588,3527.205882,null
"""Gentoo""",5484.836066,4679.741379,4587.5


In [8]:
# Оконные функции
# Добавить колонку rang_mass — ранг каждого пингвина в зависимости от массы тела внутри вида
df_penguins.with_columns(
    pl.col("body_mass_g").rank('dense', descending = True).over("species").alias("rang_mass")
).sort(by = ['species', "rang_mass"])


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,rang_mass
str,str,f64,f64,f64,f64,str,u32
"""Adelie""","""Torgersen""",null,null,null,null,null,null
"""Adelie""","""Biscoe""",43.2,19.0,197.0,4775.0,"""Male""",1
"""Adelie""","""Biscoe""",41.0,20.0,203.0,4725.0,"""Male""",2
"""Adelie""","""Torgersen""",42.9,17.6,196.0,4700.0,"""Male""",3
"""Adelie""","""Torgersen""",39.2,19.6,195.0,4675.0,"""Male""",4
…,…,…,…,…,…,…,…
"""Gentoo""","""Biscoe""",45.3,13.8,208.0,4200.0,"""Female""",44
"""Gentoo""","""Biscoe""",45.5,13.9,210.0,4200.0,"""Female""",44
"""Gentoo""","""Biscoe""",42.0,13.5,210.0,4150.0,"""Female""",45


In [9]:
# Сложная фильтрация и цепочка фильтров
# Выбрать строки, где body_mass_g > 4000 и flipper_length_mm > 200
df_penguins.filter((pl.col('body_mass_g') > 4000) & (pl.col('flipper_length_mm') > 200))


species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
str,str,f64,f64,f64,f64,str
"""Adelie""","""Dream""",41.1,18.1,205.0,4300.0,"""Male"""
"""Adelie""","""Dream""",40.8,18.9,208.0,4300.0,"""Male"""
"""Adelie""","""Biscoe""",41.0,20.0,203.0,4725.0,"""Male"""
"""Chinstrap""","""Dream""",52.0,18.1,201.0,4050.0,"""Male"""
"""Chinstrap""","""Dream""",50.5,19.6,201.0,4050.0,"""Male"""
…,…,…,…,…,…,…
"""Gentoo""","""Biscoe""",47.2,13.7,214.0,4925.0,"""Female"""
"""Gentoo""","""Biscoe""",46.8,14.3,215.0,4850.0,"""Female"""
"""Gentoo""","""Biscoe""",50.4,15.7,222.0,5750.0,"""Male"""


In [10]:
# Создать колонку heavy со значением "Yes", если body_mass_g > 5000, иначе "No".
df_penguins.with_columns(
    pl.when(pl.col('body_mass_g') > 5000)
    .then(pl.lit("Yes"))
    .otherwise(pl.lit("No"))
    .alias("heavy")
)

species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,heavy
str,str,f64,f64,f64,f64,str,str
"""Adelie""","""Torgersen""",39.1,18.7,181.0,3750.0,"""Male""","""No"""
"""Adelie""","""Torgersen""",39.5,17.4,186.0,3800.0,"""Female""","""No"""
"""Adelie""","""Torgersen""",40.3,18.0,195.0,3250.0,"""Female""","""No"""
"""Adelie""","""Torgersen""",null,null,null,null,null,"""No"""
"""Adelie""","""Torgersen""",36.7,19.3,193.0,3450.0,"""Female""","""No"""
…,…,…,…,…,…,…,…
"""Gentoo""","""Biscoe""",null,null,null,null,null,"""No"""
"""Gentoo""","""Biscoe""",46.8,14.3,215.0,4850.0,"""Female""","""No"""
"""Gentoo""","""Biscoe""",50.4,15.7,222.0,5750.0,"""Male""","""Yes"""
